# 06 — WebSocket Live Streams
## Daemon continu — collecte en temps réel des données Binance

**⚠️  LANCER CE NOTEBOOK IMMÉDIATEMENT et le maintenir actif en permanence.**
Chaque jour sans collecte live = données order book perdues définitivement.

**Streams** :
- `@aggTrade` — trades en temps réel
- `@bookTicker` — meilleur bid/ask
- `@depth5@100ms` — carnet d'ordres top 5
- `@forceOrder` — liquidations futures

**Flush** : toutes les 60 secondes vers GCS

In [ ]:
import os, sys
sys.path.insert(0, "/workspace")

from btc_pipeline.storage.gcs_client import StorageClient
from btc_pipeline.collectors.binance_live import (
    run_all_live_streams, start_live_streams_sync, STREAMS,
)

os.environ.setdefault("GCS_BUCKET_NAME", "btc-training-data")
os.environ.setdefault("STORAGE_BACKEND", "gcs")
os.environ.setdefault("WORKSPACE_DIR",  "/workspace")

storage = StorageClient()

print("Streams configurés :")
for name, cfg in STREAMS.items():
    print(f"  {name}: {cfg['base_url']}{cfg['url_suffix']}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Lancement des streams (BOUCLE INFINIE)
# ════════════════════════════════════════════════════════════════════════
# Pour arrêter : Ctrl+C ou kernel interrupt
# L'état est automatiquement sauvegardé à chaque flush

import asyncio
import nest_asyncio
nest_asyncio.apply()  # Permet asyncio dans Jupyter

stop_event = asyncio.Event()

print("▶ Démarrage de tous les streams WebSocket...")
print("  Flush toutes les 60s vers GCS")
print("  Ctrl+C pour arrêter proprement")
print()

try:
    asyncio.get_event_loop().run_until_complete(
        run_all_live_streams(storage, flush_interval=60, stop_event=stop_event)
    )
except KeyboardInterrupt:
    stop_event.set()
    print("\n⏹  Streams arrêtés proprement")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Monitoring — vérifier les données live
# ════════════════════════════════════════════════════════════════════════
from datetime import datetime, timezone
now = datetime.now(timezone.utc)
for stream in ["aggtrade", "bookticker", "depth5", "liquidations"]:
    prefix = f"raw/binance/live/{stream}/year={now.year}/month={now.month:02d}/day={now.day:02d}/"
    files = storage.list_files(prefix)
    print(f"  {stream}: {len(files)} fichiers aujourd'hui")